# In this use case we will create two MCP Servers to offer few services and will utilize those MCP Servers with MCP Client

In [4]:
# Importing Libraries
import json
import sys
from pydantic import BaseModel, Field, BeforeValidator
from typing import Dict, Any, List, Annotated

In [5]:
# Define a Universal Normalization Function
def normalize_text_input(value: Any) -> str:
    """
    Cleans up any messy string input automatically.
    - Converts to string type safely.
    - Strips leading/trailing spaces.
    - Squashes multiple spaces down to a single space (e.g., 'Souvik    Dhar' -> 'souvik dhar').
    - Converts everything to lowercase for uniform database comparison.
    """
    if not isinstance(value, str):
        value = str(value)
        
    # Remove leading/trailing spaces, lowercase it, and split/join to remove extra middle spaces
    cleaned = " ".join(value.strip().lower().split())
    return cleaned

In [6]:
# Create a reusable normalized data type
NormalizedString = Annotated[str, BeforeValidator(normalize_text_input)]

In [7]:
# Universal Protocal Packers and Formatters

# Request wrapper
def wrap_request(msg_id: int, method: str, params: dict) -> str:
    """ Pack a client action into a standardized JSON-RPC 2.0 Text Format """
    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "method": method,
        "params": params
    }
    return json.dumps(payload) # convert dictionary to raw text string

In [8]:
# Wrap Successful Response  - to be used by MCP Server
def wrap_success_response(msg_id: int, result: Any) -> str:
    """ Pack a successful server result into a standard text format 
    to avoid divercified response types from different end points. """

    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "result": result
    }

    return json.dumps(payload)

In [9]:
# Wrap Error Response - To be used by MCP Server

def wrap_error_response(msg_id: int, code: int, message: str) -> str:
    """ Pack a server error into standard JSON-RPC text format. """
    payload = {
        "jsonrpc": "2.0",
        "id": msg_id,
        "error": f"Error Code: {code}, Message: {message}"
    }
    return json.dumps(payload)

In [10]:
# Helper function for clean output
def pretty_format_json(raw_json_str: str) -> str:
    """Parses a raw JSON string and returns a beautifully indented version."""
    try:
        data = json.loads(raw_json_str)
        return json.dumps(data, indent=4) # indent=4 makes it look clean
    except Exception:
        return raw_json_str # Fallback if it's not a JSON string

In [11]:
# Below code will help to overcome Jupyter Notebook issue
def print_full_output(text: str):
    """Forces the terminal environment to print the entire text without truncation."""
    # Split the long string into lines and print them one by one
    for line in text.splitlines():
        print(line)
    # Force the terminal to instantly output everything in the queue
    sys.stdout.flush()

MCP Server 1

In [12]:
# MCP Server 1: Agents

class GitCommitInput(BaseModel):
    repository_path: NormalizedString = Field(description="The local directory path of the git repo")
    commit_message: NormalizedString = Field(description="The text summary description during commit")

class GitHubIssuesInput(BaseModel):
    repo_name: NormalizedString = Field(description="The 'organization/repository' string format. ")
    state: NormalizedString = Field(default="open", description="Filter issues by state: 'open' or 'closed' .")

In [13]:
# MCPServer1
class DevOpsMCPServer:
    def __init__(self):
        self.name = "DevOps-MCP-Server"
        self.tools_registry = {
            "git_commit_changes": {
                "description": "Stages all modified local files and saves a new Git commit. ",
                "schema": GitCommitInput.model_json_schema(),
                "model_class": GitCommitInput
            },

            "get_github_issues": {
                "description": "Queries the repository tracking board for developers",
                "schema": GitHubIssuesInput.model_json_schema(),
                "model_class": GitHubIssuesInput
            }
        }

    def handle_incoming_request(self, raw_json_request: Any) -> str:
        req_id = 0
        try:
            if isinstance(raw_json_request, str):
                request = json.loads(raw_json_request)
            else:
                request = raw_json_request
            #request = json.dumps(raw_json_request)

            req_id = request.get("id", 0)         # default is 0
            method = request.get("method", "")    # default blank method
            params = request.get("params", {})    # default blank dictionary

            # Route 1: Handle Tool Discovery Request
            
            if method == "tools/list":
                clean_menu = {}
                for name, t in self.tools_registry.items():
                    clean_menu[name] = {
                        "description": t["description"],
                        "schema": t["schema"]
                    }
                return wrap_success_response(req_id, clean_menu)
            
            if method == "tools/call":
                tool_name = params.get("name")
                tool_args = params.get("arguments", {})

                # Check if given tool is present into tools-registry
                if tool_name not in self.tools_registry:
                    return wrap_error_response(req_id, -32601, f"Tool : {tool_name} not found .")
                
                # When given tool is present

                tool_info = self.tools_registry[tool_name]
                ModelClass = tool_info["model_class"]
                validated_data = ModelClass.model_validate(tool_args)

                # Note: model_validate() is built-in function inside Pydantic which acts as a gatekeeper to enforce those rules on incoming data

                # Executing the core logic 
                if tool_name == "git_commit_changes":
                    return wrap_success_response(
                        req_id, 
                        f"[GIT] Success ! Staged Files and committed to '{validated_data.repository_path}' with message: '{validated_data.commit_message}'"
                        )
                elif tool_name == "get_github_issues":
                    mock_issues = [
                        {"id": 401, "title": "Fix memory leak in auth middleware", "status": "open"},
                        {"id": 402, "title": "Update README docs", "status": "closed"}
                    ]
                    filtered = []
                    for issue in mock_issues:
                        if issue["status"] == validated_data.state:
                            filtered.append(issue)
                    
                    response_data = {
                        "repository": validated_data.repo_name,
                        "issues": filtered
                    }

                    return wrap_success_response(
                        req_id,
                        response_data
                    )
                
            else:
                return wrap_error_response(req_id, -32601, f"Method : {method} not found")
        except Exception as e:
            return wrap_error_response(req_id, -32602, f"DevOps Server Error : {str(e)}")

MCP Server 2

In [14]:
# MCP Server 2 Agents
class EmployeeLookupInput(BaseModel):
    full_name: NormalizedString = Field(description="First and Last name of the employee to search. ")

class SlackAlertInput(BaseModel):
    channel: NormalizedString = Field(description="Target channel starting with a '#' tag. ")
    message_text: NormalizedString = Field(description="The notification text content to post to the slack channel. ")

In [15]:
# MCPServer2
class SupportMCPServer:
    def __init__(self):
        self.name = "Support-And-Comm-Server"
        self.tools_registry = {
            "lookup_employee_email": {
                "description": "Searches the internal corporate database directory for an employee's email. ",
                "schema": EmployeeLookupInput.model_json_schema(),
                "model_class": EmployeeLookupInput
            },

            "send_slack_alert": {
                "description": "Dispatches an automated instant message notification to a specific team room.",
                "schema": SlackAlertInput.model_json_schema(),
                "model_class": SlackAlertInput
            }
        }

    def handle_incoming_request(self, raw_json_request: Any) -> str:
        req_id = 0
        try:
            if isinstance(raw_json_request, str):
                request = json.loads(raw_json_request)
            else:
                request = raw_json_request
            #request = json.dumps(raw_json_request)

            req_id = request.get("id", 0)         # default is 0
            method = request.get("method", "")    # default blank method
            params = request.get("params", {})    # default blank dictionary

            # Route 1: Handle Tool Discovery Request
            
            if method == "tools/list":
                clean_menu = {}
                for name, t in self.tools_registry.items():
                    clean_menu[name] = {
                        "description": t["description"],
                        "schema": t["schema"]
                    }
                return wrap_success_response(req_id, clean_menu)
            
            if method == "tools/call":
                tool_name = params.get("name")
                tool_args = params.get("arguments", {})

                # Check if given tool is present into tools-registry
                if tool_name not in self.tools_registry:
                    return wrap_error_response(req_id, -32601, f"Tool : {tool_name} not found .")
                
                # When given tool is present

                tool_info = self.tools_registry[tool_name]
                ModelClass = tool_info["model_class"]
                validated_data = ModelClass.model_validate(tool_args)

                # Note: model_validate() is built-in function inside Pydantic which acts as a gatekeeper to enforce those rules on incoming data

                # Executing the core logic 
                if tool_name == "lookup_employee_email":
                    mock_directory = {
                        "souvik dhar": "souvik.dhar@hotmail.com", 
                        "jane smith" : "jsmith@gmail.com",
                        "tom alen" : "talen@outlook.com"
                    }
                    search_query = validated_data.full_name.lower()
                    email = mock_directory.get(search_query, "Email profile not found.")
                    return wrap_success_response(
                        req_id, 
                        f"[DIRECTORY] Result : {email}"
                        )
                elif tool_name == "send_slack_alert":                
                    return wrap_success_response(
                        req_id,
                        f"[SLACK] Message successfully delivered to channel '{validated_data.channel}': '{validated_data.message_text}' "
                    )
                
            else:
                return wrap_error_response(req_id, -32601, f"Method : {method} not found")
        except Exception as e:
            return wrap_error_response(req_id, -32602, f"SupportMCPServer Server Error : {str(e)}")

MCP Client

In [16]:
class MasterMCPClient():
    def __init__(self):
        # now client will maintain an active list of all connected servers
        self.connected_servers = []
        self.request_counter = 1

        # a master index mapping tool, names back to their home servers for smart routing
        self.tool_routing_table = {}

    
    # Defining the servers to get connected
    def connect_server(self, server_instance: Any):
        """ Registers and independent server into the client workspace. """
        self.connected_servers.append(server_instance)
        print(f"[Master Client] Successfully hooked up server instance : '{server_instance.name}")
    
    def discover_all_tools(self):
        """Queries every single registered server to assemble a master menu catalog."""
        print("\n" + "="*70)
        print(" 🔍 RUNNING NETWORK-WIDE TOOL DISCOVERY ACROSS ALL SERVERS")
        print("="*70)
        
        for server in self.connected_servers:
            msg_id = self.request_counter
            self.request_counter += 1

            # Request tools from this specific server
            raw_request = wrap_request(msg_id=msg_id, method="tools/list", params={})
            raw_response = server.handle_incoming_request(raw_request)
            
            # Parse response
            response_dict = json.loads(raw_response)
            tools_found = response_dict.get("result", {})

            print(f"\n📡 Discovered {len(tools_found)} tools from server [{server.name}]:")
            
            for tool_name, tool_details in tools_found.items():
                print(f"  🔹 Tool: {tool_name}")
                print(f"     Description: {tool_details.get('description')}")
                
                # CRITICAL STEP: Remember which server owns this tool name!
                self.tool_routing_table[tool_name] = server
                
        print("="*70 + "\n")

    def execute_tool(self, tool_name: str, arguments: dict):
        """Intelligently routes the execution request to the correct home server."""
        print("\n" + "#"*70)
        print(f" 🚀 MASTER ROUTER TRIGGERED FOR TOOL: {tool_name}")
        print("#"*70)
        
        # 1. Look up which server owns this tool in our routing map index
        target_server = self.tool_routing_table.get(tool_name)
        
        if not target_server:
            print(f" ❌ Error: Tool '{tool_name}' cannot be found on any connected server node.")
            print("#"*70 + "\n")
            return

        print(f" -> Found matching route. Directing traffic to server: [{target_server.name}]")
        print(f" -> Packaging raw parameter values for transmission...")

        # 2. Fire the JSON-RPC packet over to the specific destination server
        msg_id = self.request_counter
        self.request_counter += 1

        params_payload = {"name": tool_name, "arguments": arguments}

        # Sending request to the server
        raw_request = wrap_request(msg_id=msg_id, method="tools/call", params=params_payload)

        raw_response = target_server.handle_incoming_request(raw_request)

        # 3. Output results
        print("\n📥 TARGET SERVER RESPONSE RECEIVED:")
        print_full_output(pretty_format_json(raw_response))
        print("#"*70 + "\n")

Main Function

Simple Main Function

In [17]:
if __name__=="__main__":
    # Initiate two MCP servers
    devops_node = DevOpsMCPServer()
    support_node = SupportMCPServer()

    #Initiate the MCP Client
    client = MasterMCPClient()

    # Hook the servers up to the client infrastructure

    client.connect_server(devops_node)
    client.connect_server(support_node)

    # Run the Tools Catalog/menu list
    client.discover_all_tools()

    # Calling Git Tool from server-1
    client.execute_tool(
        "git_commit_changes", 
        {
            "repository_path": "/var/www/html/api",
            "commit_message": "docs: patch critical typo in routing documentation"
        }
    )

    # Calling Slack Tool from Server 2
    client.execute_tool(
        "send_slack_alert", 
        {
            "channel": "#ops-alerts",
            "message_text": "CRITICAL: Automated patch script finished applying code commits."
        }
    )

    # Calling Employee email lookup tool
    client.execute_tool(
        "lookup_employee_email", 
        {
            "full_name": "Souvik Dhar"
        }
    )
    

[Master Client] Successfully hooked up server instance : 'DevOps-MCP-Server
[Master Client] Successfully hooked up server instance : 'Support-And-Comm-Server

 🔍 RUNNING NETWORK-WIDE TOOL DISCOVERY ACROSS ALL SERVERS

📡 Discovered 2 tools from server [DevOps-MCP-Server]:
  🔹 Tool: git_commit_changes
     Description: Stages all modified local files and saves a new Git commit. 
  🔹 Tool: get_github_issues
     Description: Queries the repository tracking board for developers

📡 Discovered 2 tools from server [Support-And-Comm-Server]:
  🔹 Tool: lookup_employee_email
     Description: Searches the internal corporate database directory for an employee's email. 
  🔹 Tool: send_slack_alert
     Description: Dispatches an automated instant message notification to a specific team room.


######################################################################
 🚀 MASTER ROUTER TRIGGERED FOR TOOL: git_commit_changes
######################################################################
 -> Found 

Interactive Main Function

In [18]:
if __name__=="__main__":
    # Initiate two MCP servers
    devops_node = DevOpsMCPServer()
    support_node = SupportMCPServer()

    #Initiate the MCP Client
    client = MasterMCPClient()

    # Hook the servers up to the client infrastructure

    client.connect_server(devops_node)
    client.connect_server(support_node)

    # Run the Tools Catalog/menu list
    client.discover_all_tools()


    print("\n" + "═"*60)
    print(" 🕹️  ENTERED INTERACTIVE MCP CONSOLE RUNTIME")
    print("═"*60)
    print("Type 'exit' at any prompt to terminate the session.\n")

    while True:
        available_tools = list(client.tool_routing_table.keys())

        if not available_tools:
            print("❌ No tools discovered on connected servers. Exiting system.")
            break

        print("\n--- 📋 AVAILABLE SYSTEM TOOLS ---")
        for index, tool_name in enumerate(available_tools, start=1):
            parent_server = client.tool_routing_table[tool_name]
            desc = parent_server.tools_registry[tool_name]["description"]

            print(f"[{index}] {tool_name:<25} | Node: {parent_server.name:<25}\n  Summary: {desc}")

        print("-" * 60) 

        # Get the tool selection choice for the user
        user_choice = input("\nEnter the Tool Number or Tool Name to Execute :").strip()

        if user_choice.lower() == 'exit':
            print("\nShutting down interactive MCP console context. Goodbye!")
            break 
    
        selected_tool = None 
        if user_choice.isdigit():
            idx = int(user_choice) - 1
            if 0 <= idx < len(available_tools):
                selected_tool = available_tools[idx]
        elif user_choice in available_tools:
            selected_tool = user_choice

        if not selected_tool:
            print("⚠️  Invalid tool selection choice option. Please try again.")
            continue

        target_server = client.tool_routing_table[selected_tool]
        pydantic_model = target_server.tools_registry[selected_tool]["model_class"]

        schema_properties = pydantic_model.model_fields 

        print(f"\n📝 Gathering arguments for '{selected_tool}':")
        user_arguments = {}
        cancelled = False

        for field_name, field_info in schema_properties.items():
            desc_text = field_info.description or "No description available"
        
            # Show the user the schema paramter requirement constraints
            print(f" 🔹 Field [{field_name}]: {desc_text}")

            field_input = input(f"    Enter value for {field_name}: ").strip()

            if field_input.lower() == 'exit':
                cancelled = True 
                break 

            # Handle list array parsing logic if the field expects an array/list type
            # (Example: target_servers for Ansible accepts comma separated string lines)
            if getattr(field_info.annotation, "__origin__", None) is list:
                print("    (Interpreted input field payload values as a comma-separated list)")
                if field_input:
                    user_arguments[field_name] = [item.strip() for item in field_input.split(",") if item.strip()]
                else:
                    user_arguments[field_name] = []
            else:
                user_arguments[field_name] = field_input

        if cancelled:
            print("❌ Tool call execution execution routine abandoned.")
            continue

        # Step D: Hand the inputs over to the Master Client Router for dispatch processing
        client.execute_tool(selected_tool, user_arguments)



    

[Master Client] Successfully hooked up server instance : 'DevOps-MCP-Server
[Master Client] Successfully hooked up server instance : 'Support-And-Comm-Server

 🔍 RUNNING NETWORK-WIDE TOOL DISCOVERY ACROSS ALL SERVERS

📡 Discovered 2 tools from server [DevOps-MCP-Server]:
  🔹 Tool: git_commit_changes
     Description: Stages all modified local files and saves a new Git commit. 
  🔹 Tool: get_github_issues
     Description: Queries the repository tracking board for developers

📡 Discovered 2 tools from server [Support-And-Comm-Server]:
  🔹 Tool: lookup_employee_email
     Description: Searches the internal corporate database directory for an employee's email. 
  🔹 Tool: send_slack_alert
     Description: Dispatches an automated instant message notification to a specific team room.


════════════════════════════════════════════════════════════
 🕹️  ENTERED INTERACTIVE MCP CONSOLE RUNTIME
════════════════════════════════════════════════════════════
Type 'exit' at any prompt to terminate th